# SALM Fine-tuning 06 · Thirty-five little words: keyword spotting (Speech Commands)

> **Task** one-second clip → one of 35 keywords · **Data** Speech Commands v0.02 · **Metric** accuracy ↑ (judged on error rate)
> **Result** 94.78% → **97.42%** (error 5.22% → 2.58%, −50.5%) · **Time** ~1.5 h

Part of the [SALM fine-tuning series](README.md); see [00_Overview_and_Setup](00_Overview_and_Setup.ipynb) first.

## The problem: great at sentences, shaky on single words

"yes", "no", "up", "down", "marvin", "sheila", "wow"… Speech Commands is ~105,000 one-second clips of 35 words,
recorded by thousands of people on their own microphones. A transcriber should find it easy, and it mostly does:
~94.8% with the label list in the prompt.

With accuracy already above 90%, progress is best measured as **relative reduction in the error rate**. The goal is a
reliable single-word recognizer for a closed vocabulary.

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## 1. Data: the shards are sorted by keyword

Taking a prefix of the shards to save download time (say, the first 12 of 43) yields only 10 of the 35 words, and the
model collapses to answering a single word for nearly every clip. The symptom is a training loss near 0.002 within
ten steps: nothing real is learned that fast. The prep script therefore:
- reads every shard and samples uniformly (36% of train: 30,814 clips),
- asserts that every test label occurs in train.

The baseline cell below also **shuffles the training manifest on disk**, because Lhotse's 10k-row shuffle buffer can't
mix a larger sorted file (tutorial 11 shows what happens otherwise).

In [ ]:
d = M / "speechcommands"
sh(f"{PYTHON} {FT}/prepare_benchmarks.py speechcommands", log="prep_kws.log")
sh(f"{PYTHON} {FT}/prepare_benchmarks.py librispeech", log="prep_ls.log")
import collections
c = collections.Counter(json.loads(l)["text"] for l in open(d / "train.json"))
print(len(c), "words in train; most / least frequent:", c.most_common(2), c.most_common()[-2:])

## 2. Baseline

In [ ]:
TASK = f"--task cls --max-tokens 16 --labels {d}/labels.txt"
PROMPTS = {"p1": "", "p2": f"--prompt-template {q('{context} Reply with the label only.')}",
           "p3": f"--prompt {q('Transcribe the audio.')}"}   # p3: transcribe, then map the word to a label
base = {}
for p, extra in PROMPTS.items():
    for s in ("dev", "test"):
        base[p, s] = 100 * evaluate(CKPT, d / f"{s}.json", f"kws_base_{p}_{s}", TASK, extra)["score"]
    print(f"{p}  dev {base[p, 'dev']:5.2f}%   test {base[p, 'test']:5.2f}%")
BEST_P = max(PROMPTS, key=lambda p: base[p, "dev"])  # prompt chosen on dev
BASELINE = base[BEST_P, "test"]  # its test score: the baseline fine-tuning is compared with
ORACLE = max(base[p, "test"] for p in PROMPTS)  # best prompt chosen on test: an upper bound
print(f"baseline prompt (chosen on dev): {BEST_P}; test-oracle (best prompt on test): {ORACLE:.2f}")
print(f"\nbaseline = {BASELINE:.2f}% (error {100 - BASELINE:.2f}%)")
pairs = " ".join(f"{d}/{s}.json {d}/{s}_ft.json" for s in ("train", "dev", "test"))
sh(f"{PYTHON} {FT}/set_context.py --template {q('{context} Reply with the label only.')} {pairs}")
sh(f"{PYTHON} {FT}/shuffle_manifest.py {d}/train_ft.json")

## 3. Config

In [ ]:
sh(f"{PYTHON} {FT}/epoch_batches.py {d}/train_ft.json 24000 --tokenizer {CKPT}")
CFG = WORK / "configs" / "kws.yaml"
sh(f"{PYTHON} {FT}/make_ft_config.py --checkpoint {CKPT} --prompt manifest --out {CFG} --exp-dir {EXP / 'kws'} "
   f"--train-manifest {d}/train_ft.json --val-manifest {d}/dev_ft.json --val-name kws_dev "
   f"--train-encoder --train-proj --lr 1e-4 --warmup-steps 40 --max-steps 400 "
   f"--batch-tokens 24000 --limit-train-batches 100 --limit-val-batches 50 --save-top-k -1")

## 4. Train (~3.5 s/step, ~25 min, ~2.5 epochs)

In [ ]:
sh(f"{PYTHON} {FT}/run_training.py --config {CFG} --preflight --smoke --keep-steps 300 400")

## 5. Select on dev

Expected dev accuracy: **step 300 → 97.20%**, step 400 → 97.15%. The two are within noise; dev picks step 300.

In [ ]:
dev = {}
for step in (300, 400):
    out = EXPORT / f"kws_s{step}"
    sh(f"{PYTHON} {FT}/export_checkpoint.py --exp-dir {EXP / 'kws'} --step {step} --base-checkpoint {CKPT} "
       f"--out {out} --delete-ckpt", log=f"export_kws_{step}.log")
    dev[step] = 100 * evaluate(out, d / "dev_ft.json", f"kws_s{step}_dev", TASK)["score"]
    print(f"step {step}: dev accuracy {dev[step]:.2f}%")
BEST = max(dev, key=dev.get)

## 6. Test, once

In [ ]:
MODEL = EXPORT / f"kws_s{BEST}"
ft = evaluate(MODEL, d / "test_ft.json", "kws_TEST", TASK)
guard = evaluate(MODEL, M / "librispeech" / "test_clean.json", "kws_librispeech", "--task asr --normalizer english")
FT_ACC = 100 * ft["score"]
print(f"test accuracy: base {BASELINE:.2f}% → fine-tuned {FT_ACC:.2f}%   "
      f"error {100 - BASELINE:.2f}% → {100 - FT_ACC:.2f}% ({((100 - FT_ACC) - (100 - BASELINE)) / (100 - BASELINE):+.1%})")
print(f"LibriSpeech WER {100 * guard['score']:.2f}%")
result("06 baseline test accuracy", BASELINE, 94.78, 0.5)
result("06 fine-tuned test accuracy", FT_ACC, 97.42, 0.6)

Expected results:

| | test accuracy | error rate | LibriSpeech WER |
|---|---:|---:|---:|
| base, prompt chosen on dev (p2) | 94.78% | 5.22% | 1.54% |
| **fine-tuned** | **97.42%** | **2.58%** | 1.63% |

**−50.5% relative error.**

## Tidy up

Keep the selected model; delete the other exports (~63 GB each) and any leftover training checkpoints.

In [ ]:
tidy(keep=MODEL, name="kws")

## Takeaways

1. **Never subsample by shard prefix.** Datasets are often sorted by label.
2. **Near-zero loss in the first steps means a data problem.**
3. **Near the ceiling, measure error reduction.** 94.8 → 97.4 looks small, but it's half of the mistakes.